In [2]:
import pandas as pd
from collections import defaultdict, deque

In [3]:
# Read in the data

numeric_dtypes = {
    "Best of": "Int64",
    "Rank_1": "Int64",
    "Rank_2": "Int64",
    "Pts_1": "Int64",
    "Pts_2": "Int64",
    "Odd_1": "Float64",
    "Odd_2": "Float64"
}

df = pd.read_csv("../data/wta.csv",parse_dates=["Date"],dtype=numeric_dtypes,na_values=["-", "5..5"],low_memory=False,)

In [4]:
df.head(5)
df.columns
print(f"Number of records in the original dataset: {df.shape[0]:,}")

Number of records in the original dataset: 44,716


In [5]:
# 1. Odd_1 < 0 = 111
(df["Odd_1"] < 0).sum()

# 2. Odd_2 < 0 = 113
(df["Odd_2"] < 0).sum()

# 3. Both < 0 = 110 (Intersection)
((df["Odd_1"] < 0) & (df["Odd_2"] < 0)).sum()

np.int64(110)

In [6]:
# Keep only rows where neither odd is negative (Negative odds not plausible as probability cannot be negative, treat as missing values)

df = df.loc[~((df["Odd_1"] < 0) | (df["Odd_2"] < 0))].reset_index(drop=True)

In [7]:
# Check for missing values in the Winner, Rank_1 and Rank_2 columns -- minimum target and features needed.

df["Rank_1"].isna().any()
df["Rank_2"].isna().any()
df["Odd_1"].isna().any()
df["Odd_2"].isna().any()

np.False_

In [8]:
# Ensure the data is sorted by date and drop any rows with missing dates
df = df.loc[df["Date"].notna()].sort_values("Date", kind="stable").copy()

In [9]:
# Add a margins column. 
df["Margin"] = 1 / df["Odd_1"] + 1 / df["Odd_2"] - 1

In [10]:
# Checking the minimum margin value reveals an underround across all possible outcomes for two records, which appears to be rather odd.

df.loc[df["Margin"] < 0,["Date", "Player_1", "Player_2", "Odd_1", "Odd_2", "Margin", "Winner"]]

,Date,Player_1,Player_2,Odd_1,Odd_2,Margin,Winner
33029,2021-06-30,Putintseva Y.,Pironkova T.,1.4,8.9,-0.173355,Putintseva Y.
33323,2021-08-10,Van Uytvanck A.,Vekic D.,2.5,1.8,-0.044444,Vekic D.


In [11]:
# Drop records with negative bookmaker margins

negative_margin = df["Margin"] < 0
print(f"Records dropped: {negative_margin.sum()}")

df = df.loc[~negative_margin].reset_index(drop=True)

Records dropped: 2


In [12]:
# Add a normalized implied probabilities:
implied_1 = 1 / df["Odd_1"]
implied_2 = 1 / df["Odd_2"]

df["Prob_1"] = implied_1 / (implied_1 + implied_2)
df["Prob_2"] = 1 - df["Prob_1"]


In [13]:
# REGEX based standardization of names to clean up leading and trailing whitespaces.

name_cols = ["Player_1", "Player_2", "Winner"]
df[name_cols] = df[name_cols].apply(lambda col: col.str.strip())

In [14]:
# Add 2 feature columns that consider temporal data:

# 1. Create feature column considering a player's recent performance.
# 2. Performance in the last 180 days.

# Store each player's previous (date, win/loss) results.
history = defaultdict(deque)

for side in (1, 2):
    df[f"win_rate_last180d_{side}"] = float("nan")
    df[f"win_rate_last5g_{side}"] = float("nan") # Win rate in the last 5 matches, matches in last 365 days considered.

for date, matches in df.dropna(subset=["Date"]).groupby("Date", sort=True):
    cutoff_365 = date - pd.Timedelta(days=365)
    cutoff_180 = date - pd.Timedelta(days=180)

    # Calculate features before adding any results from this date.
    for idx, match in matches.iterrows():
        for side in (1, 2):
            past = history[match[f"Player_{side}"]]

            # Discard history older than 365 days.
            while past and past[0][0] < cutoff_365:
                past.popleft()

            results_180 = [
                won for day, won in past if day >= cutoff_180
            ]
            results_last5 = [
                won for day, won in list(past)[-5:]
            ]

            if results_180:
                df.at[idx, f"win_rate_last180d_{side}"] = (
                    sum(results_180) / len(results_180)
                )

            if results_last5:
                df.at[idx, f"win_rate_last5g_{side}"] = (
                    sum(results_last5) / len(results_last5)
                )

    # Update histories after calculating every match's features that day.
    for _, match in matches.iterrows():
        for side in (1, 2):
            player = match[f"Player_{side}"]
            won = int(match["Winner"] == player)
            history[player].append((date, won))

In [15]:
print(f"Number of records after cleaning pipeline: {df.shape[0]:,}")

Number of records after cleaning pipeline: 44,595


In [16]:
df.to_csv("../data/clean_wta.csv", index=False)

In [ ]:
# TODO:

# Treat the normalized odds as the bookmaker's prediction and the higher ranked player as the baseline prediction.

# See how good my models' (logistic regression? FCNN, Gradient Boosting, RNN) predictions are, relative to the baseline and the bookmaker's.

# Rely on the following assumptions - winner prediction only, book values provided in the models that rely on this as an input feature.

# Perhaps provide an interactive version that takes an input and spits out an outcome. In this case, might need to calculate input features based on the information provided.